### Model-1: Payment Time Prediction

We are building a model that predicts how many days a customer
will take to pay a new invoice.

In [188]:
import pandas as pd

In [189]:
df = pd.read_csv("../data/raw/invoices.csv")

In [190]:
df.head()

,invoice_id,cust_number,customer_name,sector,payment_term_days,invoice_amount,issue_date,due_date,status,actual_paid_date,days_to_payment,delay_vs_due_date,had_partial_payment_flag,is_big_ticket_spike,customer_archetype_TRUE_LABEL
0,INV700001,C1000,Textiles Client 001,Textiles & Apparel,60,326004.92,2024-02-26,2024-04-26,closed,2024-05-02,66.0,6.0,False,False,average_payer
1,INV700002,C1000,Textiles Client 001,Textiles & Apparel,60,194103.72,2024-03-22,2024-05-21,closed,2024-06-01,71.0,11.0,False,False,average_payer
2,INV700003,C1000,Textiles Client 001,Textiles & Apparel,60,331123.89,2024-03-25,2024-05-24,closed,2024-05-31,67.0,7.0,False,False,average_payer
3,INV700004,C1000,Textiles Client 001,Textiles & Apparel,60,191612.42,2024-04-27,2024-06-26,closed,2024-06-28,62.0,2.0,False,False,average_payer
4,INV700005,C1000,Textiles Client 001,Textiles & Apparel,60,305369.92,2024-04-30,2024-06-29,closed,2024-07-12,73.0,13.0,True,False,average_payer


In [191]:
df.shape

(5305, 15)

In [192]:
df.columns

Index(['invoice_id', 'cust_number', 'customer_name', 'sector',
       'payment_term_days', 'invoice_amount', 'issue_date', 'due_date',
       'status', 'actual_paid_date', 'days_to_payment', 'delay_vs_due_date',
       'had_partial_payment_flag', 'is_big_ticket_spike',
       'customer_archetype_TRUE_LABEL'],
      dtype='str')

In [193]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5305 entries, 0 to 5304
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   invoice_id                     5305 non-null   str    
 1   cust_number                    5305 non-null   str    
 2   customer_name                  5305 non-null   str    
 3   sector                         5305 non-null   str    
 4   payment_term_days              5305 non-null   int64  
 5   invoice_amount                 5305 non-null   float64
 6   issue_date                     5305 non-null   str    
 7   due_date                       5305 non-null   str    
 8   status                         5305 non-null   str    
 9   actual_paid_date               4958 non-null   str    
 10  days_to_payment                4958 non-null   float64
 11  delay_vs_due_date              4958 non-null   float64
 12  had_partial_payment_flag       5305 non-null   bool   
 13 

In [194]:
df.isnull().sum()

invoice_id                         0
cust_number                        0
customer_name                      0
sector                             0
payment_term_days                  0
invoice_amount                     0
issue_date                         0
due_date                           0
status                             0
actual_paid_date                 347
days_to_payment                  347
delay_vs_due_date                347
had_partial_payment_flag           0
is_big_ticket_spike                0
customer_archetype_TRUE_LABEL      0
dtype: int64

In [195]:
df["invoice_id"].duplicated().sum()

np.int64(0)

In [196]:
df["days_to_payment"].describe()

count    4958.000000
mean       50.341872
std        21.160060
min         1.000000
25%        35.000000
50%        47.000000
75%        62.000000
max       195.000000
Name: days_to_payment, dtype: float64

In [197]:
df["status"].value_counts(dropna=False)

status
closed           4958
open              264
disputed_open      83
Name: count, dtype: int64

In [198]:
df["days_to_payment"].isna().sum()

np.int64(347)

In [199]:
df["actual_paid_date"].isna().sum()

np.int64(347)

In [200]:
df[["issue_date", "due_date", "actual_paid_date"]].head(10)

,issue_date,due_date,actual_paid_date
0,2024-02-26,2024-04-26,2024-05-02
1,2024-03-22,2024-05-21,2024-06-01
2,2024-03-25,2024-05-24,2024-05-31
3,2024-04-27,2024-06-26,2024-06-28
4,2024-04-30,2024-06-29,2024-07-12
5,2024-06-22,2024-08-21,2024-08-29
6,2024-06-30,2024-08-29,2024-09-06
7,2024-07-28,2024-09-26,2024-10-05
8,2024-08-07,2024-10-06,2024-10-17
9,2024-08-09,2024-10-08,2024-10-25


In [201]:
df["days_to_payment"].describe()

count    4958.000000
mean       50.341872
std        21.160060
min         1.000000
25%        35.000000
50%        47.000000
75%        62.000000
max       195.000000
Name: days_to_payment, dtype: float64

In [202]:
(df["days_to_payment"] < 0).sum()

np.int64(0)

In [203]:
date_columns = ["issue_date", "due_date", "actual_paid_date"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col])

In [204]:
calculated_days = (
    df["actual_paid_date"] - df["issue_date"]
).dt.days

In [205]:
(df["days_to_payment"] - calculated_days).abs().dropna().max()

np.float64(0.0)

In [206]:
training_df = df[df["status"] == "closed"].copy()

In [207]:
training_df.shape

(4958, 15)

In [208]:
training_df["status"].value_counts()

status
closed    4958
Name: count, dtype: int64

In [209]:
training_df["days_to_payment"].isna().sum()

np.int64(0)

In [210]:
training_df.columns.tolist()

['invoice_id',
 'cust_number',
 'customer_name',
 'sector',
 'payment_term_days',
 'invoice_amount',
 'issue_date',
 'due_date',
 'status',
 'actual_paid_date',
 'days_to_payment',
 'delay_vs_due_date',
 'had_partial_payment_flag',
 'is_big_ticket_spike',
 'customer_archetype_TRUE_LABEL']

In [211]:
training_df["is_big_ticket_spike"].value_counts(dropna=False)

is_big_ticket_spike
False    4745
True      213
Name: count, dtype: int64

In [212]:
training_df["customer_archetype_TRUE_LABEL"].value_counts(dropna=False)

customer_archetype_TRUE_LABEL
average_payer          1521
prompt_payer           1224
seasonal_payer          554
erratic_payer           496
deteriorating_payer     406
improving_payer         363
chronic_late_payer      344
cold_start               50
Name: count, dtype: int64

In [213]:
training_df.groupby("cust_number")["issue_date"].count().describe()

count    180.000000
mean      27.544444
std       13.655507
min        2.000000
25%       17.000000
50%       28.500000
75%       38.000000
max       53.000000
Name: issue_date, dtype: float64

In [214]:
training_df.sort_values(["cust_number", "issue_date"])[
    ["cust_number", "issue_date", "days_to_payment"]
].head(20)

,cust_number,issue_date,days_to_payment
0,C1000,2024-02-26,66.0
1,C1000,2024-03-22,71.0
2,C1000,2024-03-25,67.0
3,C1000,2024-04-27,62.0
4,C1000,2024-04-30,73.0
5,C1000,2024-06-22,68.0
6,C1000,2024-06-30,68.0
7,C1000,2024-07-28,69.0
8,C1000,2024-08-07,71.0
9,C1000,2024-08-09,77.0


In [215]:
historical_df = training_df.sort_values(
    ["cust_number", "issue_date"]
).copy()

In [216]:
historical_df["previous_payment_days"] = (
    historical_df.groupby("cust_number")["days_to_payment"]
    .shift(1)
)

In [217]:
historical_df[
    ["cust_number", "issue_date", "days_to_payment", "previous_payment_days"]
].head(10)

,cust_number,issue_date,days_to_payment,previous_payment_days
0,C1000,2024-02-26,66.0,NaN
1,C1000,2024-03-22,71.0,66.0
2,C1000,2024-03-25,67.0,71.0
3,C1000,2024-04-27,62.0,67.0
4,C1000,2024-04-30,73.0,62.0
5,C1000,2024-06-22,68.0,73.0
6,C1000,2024-06-30,68.0,68.0
7,C1000,2024-07-28,69.0,68.0
8,C1000,2024-08-07,71.0,69.0
9,C1000,2024-08-09,77.0,71.0


In [218]:
historical_df["customer_avg_payment_days"] = (
    historical_df.groupby("cust_number")["days_to_payment"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [219]:
historical_df[
    [
        "cust_number",
        "issue_date",
        "days_to_payment",
        "previous_payment_days",
        "customer_avg_payment_days"
    ]
].head(10)

,cust_number,issue_date,days_to_payment,previous_payment_days,customer_avg_payment_days
0,C1000,2024-02-26,66.0,NaN,NaN
1,C1000,2024-03-22,71.0,66.0,66.000000
2,C1000,2024-03-25,67.0,71.0,68.500000
3,C1000,2024-04-27,62.0,67.0,68.000000
4,C1000,2024-04-30,73.0,62.0,66.500000
5,C1000,2024-06-22,68.0,73.0,67.800000
6,C1000,2024-06-30,68.0,68.0,67.833333
7,C1000,2024-07-28,69.0,68.0,67.857143
8,C1000,2024-08-07,71.0,69.0,68.000000
9,C1000,2024-08-09,77.0,71.0,68.333333


In [220]:
historical_df["customer_avg_payment_days"].isna().sum()

np.int64(180)

In [221]:
historical_df["customer_recent_avg_payment_days"] = (
    historical_df.groupby("cust_number")["days_to_payment"]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

In [222]:
historical_df[
    [
        "cust_number",
        "issue_date",
        "days_to_payment",
        "customer_avg_payment_days",
        "customer_recent_avg_payment_days"
    ]
].head(10)

,cust_number,issue_date,days_to_payment,customer_avg_payment_days,customer_recent_avg_payment_days
0,C1000,2024-02-26,66.0,NaN,NaN
1,C1000,2024-03-22,71.0,66.000000,66.000000
2,C1000,2024-03-25,67.0,68.500000,68.500000
3,C1000,2024-04-27,62.0,68.000000,68.000000
4,C1000,2024-04-30,73.0,66.500000,66.666667
5,C1000,2024-06-22,68.0,67.800000,67.333333
6,C1000,2024-06-30,68.0,67.833333,67.666667
7,C1000,2024-07-28,69.0,67.857143,69.666667
8,C1000,2024-08-07,71.0,68.000000,68.333333
9,C1000,2024-08-09,77.0,68.333333,69.333333


In [223]:
historical_df["customer_invoice_count"] = (
    historical_df.groupby("cust_number").cumcount()
)

In [224]:
historical_df[
    [
        "cust_number",
        "issue_date",
        "days_to_payment",
        "customer_invoice_count"
    ]
].head(10)

,cust_number,issue_date,days_to_payment,customer_invoice_count
0,C1000,2024-02-26,66.0,0
1,C1000,2024-03-22,71.0,1
2,C1000,2024-03-25,67.0,2
3,C1000,2024-04-27,62.0,3
4,C1000,2024-04-30,73.0,4
5,C1000,2024-06-22,68.0,5
6,C1000,2024-06-30,68.0,6
7,C1000,2024-07-28,69.0,7
8,C1000,2024-08-07,71.0,8
9,C1000,2024-08-09,77.0,9


In [225]:
historical_df["customer_payment_std"] = (
    historical_df.groupby("cust_number")["days_to_payment"]
    .transform(lambda x: x.shift(1).expanding().std())
)

In [226]:
historical_df[
    [
        "cust_number",
        "issue_date",
        "days_to_payment",
        "customer_invoice_count",
        "customer_payment_std"
    ]
].head(10)

,cust_number,issue_date,days_to_payment,customer_invoice_count,customer_payment_std
0,C1000,2024-02-26,66.0,0,NaN
1,C1000,2024-03-22,71.0,1,NaN
2,C1000,2024-03-25,67.0,2,3.535534
3,C1000,2024-04-27,62.0,3,2.645751
4,C1000,2024-04-30,73.0,4,3.696846
5,C1000,2024-06-22,68.0,5,4.324350
6,C1000,2024-06-30,68.0,6,3.868678
7,C1000,2024-07-28,69.0,7,3.532165
8,C1000,2024-08-07,71.0,8,3.295018
9,C1000,2024-08-09,77.0,9,3.240370


In [227]:
historical_df["payment_behavior_trend"] = (
    historical_df["customer_recent_avg_payment_days"]
    - historical_df["customer_avg_payment_days"]
)

In [228]:
historical_df[
    [
        "cust_number",
        "issue_date",
        "customer_avg_payment_days",
        "customer_recent_avg_payment_days",
        "payment_behavior_trend"
    ]
].head(10)

,cust_number,issue_date,customer_avg_payment_days,customer_recent_avg_payment_days,payment_behavior_trend
0,C1000,2024-02-26,NaN,NaN,NaN
1,C1000,2024-03-22,66.000000,66.000000,0.000000
2,C1000,2024-03-25,68.500000,68.500000,0.000000
3,C1000,2024-04-27,68.000000,68.000000,0.000000
4,C1000,2024-04-30,66.500000,66.666667,0.166667
5,C1000,2024-06-22,67.800000,67.333333,-0.466667
6,C1000,2024-06-30,67.833333,67.666667,-0.166667
7,C1000,2024-07-28,67.857143,69.666667,1.809524
8,C1000,2024-08-07,68.000000,68.333333,0.333333
9,C1000,2024-08-09,68.333333,69.333333,1.000000


In [229]:
historical_df["invoice_amount"].describe()

count    4.958000e+03
mean     2.125292e+05
std      4.161850e+05
min      3.000000e+03
25%      4.509367e+04
50%      9.158992e+04
75%      1.930326e+05
max      1.055864e+07
Name: invoice_amount, dtype: float64

In [230]:
historical_df["invoice_amount"].isna().sum()

np.int64(0)

In [231]:
historical_df["invoice_amount"].nunique()

4950

In [232]:
historical_df[["invoice_amount", "days_to_payment"]].corr()

,invoice_amount,days_to_payment
invoice_amount,1.000000,0.028258
days_to_payment,0.028258,1.000000


In [233]:
historical_df["payment_term_days"].describe()

count    4958.000000
mean       40.746269
std        15.704098
min        15.000000
25%        30.000000
50%        45.000000
75%        45.000000
max        90.000000
Name: payment_term_days, dtype: float64

In [234]:
historical_df["payment_term_days"].value_counts().sort_index()


payment_term_days
15     346
30    2026
45    1644
60     757
90     185
Name: count, dtype: int64

In [235]:
historical_df["sector"].value_counts()

sector
Retail & Distribution        751
Chemicals & Plastics         646
Auto Components              592
FMCG & Food Processing       556
Metal & Engineering          498
Construction & Infra         479
Pharma & Healthcare          439
Textiles & Apparel           424
IT & Business Services       308
Electronics & Electricals    265
Name: count, dtype: int64

In [236]:
historical_df["days_to_payment"].median()

np.float64(47.0)

In [237]:
historical_df["days_to_payment"].mean()

np.float64(50.34187172246874)

In [238]:
historical_df["issue_date"].min()

Timestamp('2024-02-14 00:00:00')

In [239]:
historical_df["issue_date"].max()

Timestamp('2026-07-16 00:00:00')

In [240]:
id="e5h1c2"
dates = historical_df["issue_date"].sort_values().reset_index(drop=True)

train_end = dates.iloc[int(len(dates) * 0.70)]
validation_end = dates.iloc[int(len(dates) * 0.85)]

train_end, validation_end

(Timestamp('2025-10-05 00:00:00'), Timestamp('2026-02-14 00:00:00'))

In [241]:
train_df = historical_df[
    historical_df["issue_date"] <= train_end
].copy()

validation_df = historical_df[
    (historical_df["issue_date"] > train_end) &
    (historical_df["issue_date"] <= validation_end)
].copy()

test_df = historical_df[
    historical_df["issue_date"] > validation_end
].copy()

In [242]:
train_df.shape, validation_df.shape, test_df.shape

((3473, 21), (745, 21), (740, 21))

In [243]:
train_df["issue_date"].min(), train_df["issue_date"].max()

(Timestamp('2024-02-14 00:00:00'), Timestamp('2025-10-05 00:00:00'))

In [244]:
validation_df["issue_date"].min(), validation_df["issue_date"].max()

(Timestamp('2025-10-06 00:00:00'), Timestamp('2026-02-14 00:00:00'))

In [245]:
test_df["issue_date"].min(), test_df["issue_date"].max()

(Timestamp('2026-02-15 00:00:00'), Timestamp('2026-07-16 00:00:00'))

In [246]:
historical_df[
    ["cust_number", "issue_date"]
].head(20)

,cust_number,issue_date
0,C1000,2024-02-26
1,C1000,2024-03-22
2,C1000,2024-03-25
3,C1000,2024-04-27
4,C1000,2024-04-30
5,C1000,2024-06-22
6,C1000,2024-06-30
7,C1000,2024-07-28
8,C1000,2024-08-07
9,C1000,2024-08-09


In [247]:
historical_df.groupby("cust_number")["issue_date"].apply(
    lambda x: x.is_monotonic_increasing
).value_counts()

issue_date
True    180
Name: count, dtype: int64

In [248]:
historical_df.columns.tolist()

['invoice_id',
 'cust_number',
 'customer_name',
 'sector',
 'payment_term_days',
 'invoice_amount',
 'issue_date',
 'due_date',
 'status',
 'actual_paid_date',
 'days_to_payment',
 'delay_vs_due_date',
 'had_partial_payment_flag',
 'is_big_ticket_spike',
 'customer_archetype_TRUE_LABEL',
 'previous_payment_days',
 'customer_avg_payment_days',
 'customer_recent_avg_payment_days',
 'customer_invoice_count',
 'customer_payment_std',
 'payment_behavior_trend']

In [249]:
feature_columns = [
    "customer_avg_payment_days",
    "customer_recent_avg_payment_days",
    "customer_invoice_count",
    "customer_payment_std",
    "payment_behavior_trend",
    "invoice_amount",
    "payment_term_days",
    "sector"
]

historical_df[feature_columns].isna().sum()

customer_avg_payment_days           180
customer_recent_avg_payment_days    180
customer_invoice_count                0
customer_payment_std                360
payment_behavior_trend              180
invoice_amount                        0
payment_term_days                     0
sector                                0
dtype: int64

In [250]:
historical_features = [
    "customer_avg_payment_days",
    "customer_recent_avg_payment_days",
    "customer_payment_std",
    "payment_behavior_trend"
]

train_df[historical_features].median()

customer_avg_payment_days           47.500000
customer_recent_avg_payment_days    47.333333
customer_payment_std                 4.983903
payment_behavior_trend               0.000000
dtype: float64

Model 1 - Final Feature Dataset

In [251]:
id_columns = ["invoice_id", "cust_number", "issue_date"]

model1_features = [
    "customer_avg_payment_days",
    "customer_recent_avg_payment_days",
    "customer_invoice_count",
    "customer_payment_std",
    "payment_behavior_trend",
    "previous_payment_days",      # <-- added
    "invoice_amount",
    "payment_term_days",
    "sector"
]

target = "days_to_payment"
diagnostic_only = ["customer_archetype_TRUE_LABEL"]   # kept for analysis, NOT for training

model1_df = historical_df[
    id_columns + model1_features + [target] + diagnostic_only
].copy()

model1_df.to_csv("../data/processed/model1_features.csv", index=False)

In [252]:
model1_df.shape

(4958, 14)

In [253]:
model1_df.head()

,invoice_id,cust_number,issue_date,customer_avg_payment_days,customer_recent_avg_payment_days,customer_invoice_count,customer_payment_std,payment_behavior_trend,previous_payment_days,invoice_amount,payment_term_days,sector,days_to_payment,customer_archetype_TRUE_LABEL
0,INV700001,C1000,2024-02-26,NaN,NaN,0,NaN,NaN,NaN,326004.92,60,Textiles & Apparel,66.0,average_payer
1,INV700002,C1000,2024-03-22,66.0,66.000000,1,NaN,0.000000,66.0,194103.72,60,Textiles & Apparel,71.0,average_payer
2,INV700003,C1000,2024-03-25,68.5,68.500000,2,3.535534,0.000000,71.0,331123.89,60,Textiles & Apparel,67.0,average_payer
3,INV700004,C1000,2024-04-27,68.0,68.000000,3,2.645751,0.000000,67.0,191612.42,60,Textiles & Apparel,62.0,average_payer
4,INV700005,C1000,2024-04-30,66.5,66.666667,4,3.696846,0.166667,62.0,305369.92,60,Textiles & Apparel,73.0,average_payer


In [254]:
model1_df[
    ["invoice_amount", "payment_term_days", "sector", "days_to_payment"]
].isna().sum()

invoice_amount       0
payment_term_days    0
sector               0
days_to_payment      0
dtype: int64

In [255]:
model1_df.to_csv(
    "../data/processed/model1_features.csv",
    index=False
)

In [256]:
check_df = pd.read_csv("../data/processed/model1_features.csv")

In [257]:
check_df.shape

(4958, 14)

In [258]:
check_df.head()

,invoice_id,cust_number,issue_date,customer_avg_payment_days,customer_recent_avg_payment_days,customer_invoice_count,customer_payment_std,payment_behavior_trend,previous_payment_days,invoice_amount,payment_term_days,sector,days_to_payment,customer_archetype_TRUE_LABEL
0,INV700001,C1000,2024-02-26,NaN,NaN,0,NaN,NaN,NaN,326004.92,60,Textiles & Apparel,66.0,average_payer
1,INV700002,C1000,2024-03-22,66.0,66.000000,1,NaN,0.000000,66.0,194103.72,60,Textiles & Apparel,71.0,average_payer
2,INV700003,C1000,2024-03-25,68.5,68.500000,2,3.535534,0.000000,71.0,331123.89,60,Textiles & Apparel,67.0,average_payer
3,INV700004,C1000,2024-04-27,68.0,68.000000,3,2.645751,0.000000,67.0,191612.42,60,Textiles & Apparel,62.0,average_payer
4,INV700005,C1000,2024-04-30,66.5,66.666667,4,3.696846,0.166667,62.0,305369.92,60,Textiles & Apparel,73.0,average_payer


In [259]:
model1_df[target].describe()

count    4958.000000
mean       50.341872
std        21.160060
min         1.000000
25%        35.000000
50%        47.000000
75%        62.000000
max       195.000000
Name: days_to_payment, dtype: float64

In [260]:
model1_df[model1_features].isna().sum()

customer_avg_payment_days           180
customer_recent_avg_payment_days    180
customer_invoice_count                0
customer_payment_std                360
payment_behavior_trend              180
previous_payment_days               180
invoice_amount                        0
payment_term_days                     0
sector                                0
dtype: int64